# Apigee Template: REST-AI-Messages

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-template-repository/blob/main/notebooks/REST-AI-Messages.ipynb)

**Template Name:** `REST-AI-Messages`  
**Description:** Anthropic Claude Messages API proxy (`/v1/messages`) supporting native Anthropic payloads, multi-turn messages, streaming responses, intelligent model routing to Google Cloud Vertex AI, model checks, and token usage analytics.

### Key Capabilities:
- **Native Anthropic `/v1/messages` Protocol:** Accept standard Anthropic messages requests with system prompts, multi-turn messages, and streaming.
- **Vertex AI Model Garden Integration:** Apigee proxies requests directly to Google Cloud Vertex AI (`publishers/anthropic/models/{model}:rawPredict` or `:streamRawPredict`), automatically injects `anthropic_version: "vertex-2023-10-16"`, and mints Google Cloud OAuth tokens via `AM-SetGoogleToken`.
- **Model Routing & Checks:** Routes Anthropic Claude models (`anthropic/*` or `claude-*`) to Vertex AI Model Garden, while also supporting Google Gemini models via `googlecloud-oai`.
- **Token Usage Analytics & Data Collection:** Intercepts response `usage` (`input_tokens`, `output_tokens`) and logs them into Apigee Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`, `dc_ai_model`). These metrics populate Apigee Custom Reports for real-time cost, throughput, and token consumption tracking.

### Documentation & References:
- [Anthropic Messages API Reference](https://docs.anthropic.com/en/api/messages)
- [Google Cloud Vertex AI - Anthropic Claude Models](https://cloud.google.com/vertex-ai/generative-ai/docs/partner-models/use-claude)
- [Apigee Feature Templater (aft) GitHub](https://github.com/apigee/apigee-templater)
- [Apigee Data Collectors Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/data-collectors)
- [Apigee Custom Reports Documentation](https://cloud.google.com/apigee/docs/api-platform/analytics/reports-overview)

### Workflow:
1. **Configuration & Authentication:** Enter your `GOOGLE_CLOUD_PROJECT`, `APIGEE_ENV`, and optional API credentials.
2. **Setup, Initialize Resources & Deploy:** Install `aft`, run `sh/initialize.sh` (service account, IAM roles, data collectors, reports), resolve `APIGEE_HOST`, and deploy `REST-AI-Messages.yaml`.
3. **Test Initialized Messages API:** Send standard unary requests, multi-turn conversations, and streaming requests using `APIGEE_HOST`.

---

## 1. Configuration & Authentication

Specify your Google Cloud Project ID (Apigee Organization) and target Apigee environment.

In [ ]:
# @title 1. Configuration & Authentication
import os

# @markdown Enter your Google Cloud Project ID (Apigee Organization):
GOOGLE_CLOUD_PROJECT = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
ANTHROPIC_API_KEY = ""  # @param {type:"string"}

os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ORG"] = GOOGLE_CLOUD_PROJECT
os.environ["APIGEE_ENV"] = APIGEE_ENV
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
os.environ["APIGEE_SA"] = f"apigee-service@{GOOGLE_CLOUD_PROJECT}.iam.gserviceaccount.com"

try:
    from google.colab import auth
    auth.authenticate_user()
    print(f"Authenticated with Google Cloud for project: {GOOGLE_CLOUD_PROJECT}")
except ImportError:
    print("Running outside Google Colab.")


## 2. Setup Tools, Initialize Resources & Deploy Template

Downloads `templates/REST-AI-Messages.yaml` and `sh/initialize.sh` (if running standalone in Colab), installs `aft`, runs resource initialization (service accounts, IAM bindings, data collectors, custom reports), sets `APIGEE_HOST`, and deploys the template.

In [ ]:
# @title 2. Setup, Initialize & Deploy Template
import os
import subprocess

# 1. Install Apigee Feature Templater (aft) CLI if needed
!which aft >/dev/null 2>&1 || curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh

# 2. Download template and initialize script if running standalone
REPO_RAW = "https://raw.githubusercontent.com/gcp-samples/apigee-template-repository/main"
TEMPLATE_FILE = "REST-AI-Messages.yaml" if os.path.exists("REST-AI-Messages.yaml") else "templates/REST-AI-Messages.yaml" if os.path.exists("templates/REST-AI-Messages.yaml") else "REST-AI-Messages.yaml"
os.environ["TEMPLATE_FILE"] = TEMPLATE_FILE

if not os.path.exists(TEMPLATE_FILE):
    !curl -fsSL -O {REPO_RAW}/templates/REST-AI-Messages.yaml

if not os.path.exists("sh/initialize.sh"):
    !mkdir -p sh && curl -fsSL -o sh/initialize.sh {REPO_RAW}/sh/initialize.sh

# 3. Run initialization script (sets up service account, IAM bindings, data collectors, reports)
!bash sh/initialize.sh

# 4. Resolve APIGEE_HOST using aft describe
cmd = 'aft describe --project "$GOOGLE_CLOUD_PROJECT" -f json | jq --raw-output ".environmentGroups[] | select(any(.attachments[]; .environment == \"$APIGEE_ENV\")) | .hostnames[0]"'
try:
    host = subprocess.check_output(cmd, shell=True, text=True).strip()
    if host and host != "null":
        os.environ["APIGEE_HOST"] = host
        print(f"APIGEE_HOST resolved to: {host}")
except Exception as e:
    print(f"Could not automatically resolve APIGEE_HOST via aft: {e}")

# 5. Deploy template with aft
!aft "$TEMPLATE_FILE" \
  --project="$GOOGLE_CLOUD_PROJECT" \
  --env="$APIGEE_ENV" \
  --sa="$APIGEE_SA"


## 3. Test Initialized Messages API via APIGEE_HOST

Send requests to `https://${APIGEE_HOST}/v1/messages`.

- **Model Routing & Vertex AI Integration:** Apigee inspects the `model` parameter, adapts the payload with `anthropic_version: "vertex-2023-10-16"`, and forwards it to Vertex AI Model Garden (`publishers/anthropic/models/{model}:rawPredict`).
- **IAM Token Minting:** The `AM-SetGoogleToken` policy attaches the Google Cloud service account token to backend requests, removing the need for callers to provide Google Cloud credentials.
- **Token Usage Recording:** Apigee intercepts response `usage` (`input_tokens`, `output_tokens`) and logs them directly into Apigee Data Collectors (`dc_ai_prompt_token_count`, `dc_ai_response_token_count`, `dc_ai_total_token_count`).

In [ ]:
# @title Setup Test Client & APIGEE_HOST
import os
import json
import requests
import subprocess

PROJECT_ID = os.getenv("GOOGLE_CLOUD_PROJECT", "")
APIGEE_ENV = os.getenv("APIGEE_ENV", "dev")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

# Determine APIGEE_HOST
APIGEE_HOST = os.getenv("APIGEE_HOST")
if not APIGEE_HOST or APIGEE_HOST == "null":
    APIGEE_HOST = f"{PROJECT_ID}-{APIGEE_ENV}.apigee.net"
    os.environ["APIGEE_HOST"] = APIGEE_HOST

# Retrieve GCP access token if available for caller authorization
try:
    gcp_token = subprocess.check_output(["gcloud", "auth", "application-default", "print-access-token"], text=True).strip()
except Exception:
    try:
        gcp_token = subprocess.check_output(["gcloud", "auth", "print-access-token"], text=True).strip()
    except Exception:
        gcp_token = ""

print(f"APIGEE_HOST: {APIGEE_HOST}")
print(f"Messages Endpoint: https://{APIGEE_HOST}/v1/messages")

def send_message(model: str, messages: list, system: str = None, max_tokens: int = 1024, stream: bool = False):
    """
    Sends an Anthropic Messages request through the Apigee Gateway.
    Apigee routes the model, adapts parameters for Vertex AI, automatically
    attaches IAM credentials, and records token metrics into Apigee Data Collectors.
    """
    url = f"https://{APIGEE_HOST}/v1/messages"
    
    headers = {
        "Content-Type": "application/json",
        "anthropic-version": "2023-06-01"
    }
    if gcp_token:
        headers["Authorization"] = f"Bearer {gcp_token}"
    if ANTHROPIC_API_KEY:
        headers["x-api-key"] = ANTHROPIC_API_KEY

    payload = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "stream": stream
    }
    if system:
        payload["system"] = system

    print(f"\n---> Sending [{model}] request to {url}...")
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30, stream=stream)
        print(f"HTTP Status: {response.status_code}")
        
        if stream:
            print("Streaming Response Chunks:")
            for chunk in response.iter_lines():
                if chunk:
                    print(chunk.decode("utf-8"))
            return None
        
        try:
            data = response.json()
            print(json.dumps(data, indent=2))
            
            # Display token usage recorded by Apigee
            usage = data.get("usage", {})
            if usage:
                print("\n[Token Usage Recorded by Apigee Data Collectors]")
                print(f"  Input Tokens:  {usage.get('input_tokens', 'N/A')}")
                print(f"  Output Tokens: {usage.get('output_tokens', 'N/A')}")
            return data
        except Exception:
            print(response.text)
            return None
    except Exception as e:
        print("Request error:", e)
        return None


In [ ]:
# @title Test 1: Standard Anthropic Message Request
# Apigee routes claude models to Vertex AI Model Garden and records token analytics.
resp1 = send_message(
    model="claude-3-5-sonnet-v2@20241022",
    messages=[
        {"role": "user", "content": "Explain the role of an API gateway in enterprise AI governance in two sentences."}
    ]
)


In [ ]:
# @title Test 2: Multi-turn Conversation with System Prompt
# Validates system prompt handling and multi-turn message history preservation.
conversation = [
    {"role": "user", "content": "What is model fallback in API management?"},
    {"role": "assistant", "content": "Model fallback automatically redirects AI requests to an alternate model or provider when the primary model experiences outages, rate limits, or latency spikes."},
    {"role": "user", "content": "What is one key metric to monitor when this occurs?"}
]

resp2 = send_message(
    model="claude-3-5-haiku@20241022",
    messages=conversation,
    system="You are an expert enterprise API and AI architect. Respond concisely."
)


In [ ]:
# @title Test 3: Streaming Messages Response (stream=True)
# Proxies server-sent events (SSE) while Apigee aggregates streaming token counts.
resp3 = send_message(
    model="claude-3-5-sonnet-v2@20241022",
    messages=[
        {"role": "user", "content": "List 3 main benefits of using Apigee for AI traffic."}
    ],
    stream=True
)


## 4. Verify Apigee Analytics & Reports

Because `sh/initialize.sh` provisioned Data Collectors and Custom Reports, every request processed by `REST-AI-Messages` records token analytics:

1. Open the [Google Cloud Apigee Console](https://console.cloud.google.com/apigee).
2. Navigate to **Analytics > Custom Reports**.
3. Inspect the pre-configured reports:
   - **`ai_token_cost_by_model_user`**: Aggregated AI token cost by model and user/developer.
   - **`ai_model_usage_latency`**: Real-time latency and call volume per model.
   - **`ai_token_counts_by_model_user`**: Prompt, response, and total token count breakdowns.
